# NB-03 · Health Data Ecosystem & ICD/CPT/LOINC Code Structures
## Module 01 — Health Analytics with Python
---
**Learning objectives**
- Navigate the major public health data ecosystems (CMS, CDC, HCUP, PhysioNet)
- Decode ICD-10-CM, CPT-4, and LOINC code hierarchies in Python
- Build lookup and mapping utilities for clinical code systems
- Flag and validate code formats for data quality checks
- Simulate FHIR-like data and parse it with pandas
- Understand HIPAA safe-harbour de-identification requirements

**Estimated time:** 2 hours  
**Level:** Beginner → Intermediate  
**Libraries:** `pandas`, `numpy`, `re`, `json`, `pathlib`


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 30)
print("Setup complete.")


## 2. The Health Data Ecosystem

Understanding where data originates shapes every analytical decision — from choosing the right denominator to knowing which variables are trustworthy.


In [ ]:
# Build a reference table of major health data sources
sources = [
    {
        'Source'    : 'CMS Medicare FFS Claims',
        'Level'     : 'Patient/claim',
        'Coverage'  : 'Medicare beneficiaries (65+, disabled)',
        'Key_vars'  : 'ICD-10, DRG, revenue codes, provider NPI',
        'Access'    : 'Data use agreement (DUA)',
        'Format'    : 'CSV / SAS',
        'Latency'   : '12–18 months',
        'Use_case'  : 'Claims-based quality measures, utilisation'
    },
    {
        'Source'    : 'HCUP NIS',
        'Level'     : 'Discharge',
        'Coverage'  : 'All payers, ~7M stays/year',
        'Key_vars'  : 'ICD-10, DRG, LOS, charges, payer, APR-DRG',
        'Access'    : 'AHRQ purchase (~$300)',
        'Format'    : 'SAS / ASCII',
        'Latency'   : '~24 months',
        'Use_case'  : 'National inpatient trends, LOS, costs'
    },
    {
        'Source'    : 'CDC BRFSS',
        'Level'     : 'Survey respondent',
        'Coverage'  : '400k+ US adults/year',
        'Key_vars'  : 'Chronic conditions, BMI, smoking, state FIPS',
        'Access'    : 'Public download',
        'Format'    : 'XPT / CSV',
        'Latency'   : '~12 months',
        'Use_case'  : 'Prevalence of risk factors, state comparisons'
    },
    {
        'Source'    : 'MIMIC-IV Demo',
        'Level'     : 'Patient / ICU stay',
        'Coverage'  : 'De-identified ICU patients (BIDMC)',
        'Key_vars'  : 'Labs, vitals, meds, diagnoses, notes, procedures',
        'Access'    : 'Free; PhysioNet credentialing (~30 min)',
        'Format'    : 'CSV',
        'Latency'   : 'Historical (2008–2019)',
        'Use_case'  : 'ML model training, clinical prediction'
    },
    {
        'Source'    : 'Synthea',
        'Level'     : 'Synthetic patient',
        'Coverage'  : 'Unlimited synthetic FHIR records',
        'Key_vars'  : 'Full FHIR: encounters, labs, meds, conditions',
        'Access'    : 'Open source, run locally',
        'Format'    : 'FHIR JSON / CSV',
        'Latency'   : 'Generated on demand',
        'Use_case'  : 'Education, algorithm development (no DUA)'
    },
    {
        'Source'    : 'NYC SPARCS',
        'Level'     : 'Discharge',
        'Coverage'  : 'NY State inpatient and outpatient',
        'Key_vars'  : 'ZIP, APR-DRG, charges, payer, diagnosis',
        'Access'    : 'Public download',
        'Format'    : 'CSV',
        'Latency'   : '~12 months',
        'Use_case'  : 'Geospatial health disparities, NYC analyses'
    },
]

df_sources = pd.DataFrame(sources).set_index('Source')
print("Major health data sources:")
display(df_sources[['Level','Coverage','Access','Use_case']])


## 3. ICD-10-CM Code Structure

ICD-10-CM (International Classification of Diseases, 10th Revision, Clinical Modification) is the standard for diagnosis coding in US healthcare.

```
E   1 1 . 9
│   │ │   └── 5th-7th characters: clinical detail / laterality / episode
│   │ └────── 4th character: subcategory / anatomic site
│   └──────── 2nd-3rd characters: condition / etiology
└──────────── 1st character: chapter letter (A–Z)
```


In [ ]:
# --- 3.1  ICD-10-CM chapter map ---
icd10_chapters = {
    'A': 'Certain infectious and parasitic diseases',
    'B': 'Certain infectious and parasitic diseases',
    'C': 'Neoplasms',
    'D': 'Neoplasms / Blood diseases',
    'E': 'Endocrine, nutritional and metabolic diseases',
    'F': 'Mental, behavioural and neurodevelopmental disorders',
    'G': 'Diseases of the nervous system',
    'H': 'Diseases of the eye / ear',
    'I': 'Diseases of the circulatory system',
    'J': 'Diseases of the respiratory system',
    'K': 'Diseases of the digestive system',
    'L': 'Diseases of the skin and subcutaneous tissue',
    'M': 'Diseases of the musculoskeletal system',
    'N': 'Diseases of the genitourinary system',
    'O': 'Pregnancy, childbirth and the puerperium',
    'P': 'Conditions originating in the perinatal period',
    'Q': 'Congenital malformations and chromosomal abnormalities',
    'R': 'Symptoms, signs and abnormal clinical findings',
    'S': 'Injury, poisoning and external causes',
    'T': 'Injury, poisoning and external causes',
    'U': 'Codes for special purposes (COVID-19 etc.)',
    'V': 'External causes of morbidity',
    'W': 'External causes of morbidity',
    'X': 'External causes of morbidity',
    'Y': 'External causes of morbidity',
    'Z': 'Factors influencing health status / contact with health services',
}

chapter_df = pd.DataFrame.from_dict(
    icd10_chapters, orient='index', columns=['Chapter description']
)
chapter_df.index.name = 'First letter'
print("ICD-10-CM chapter map:")
display(chapter_df.drop_duplicates())


In [ ]:
# --- 3.2  ICD-10-CM parser ---
def parse_icd10cm(code: str) -> dict:
    """
    Parse an ICD-10-CM code into its structural components.
    Returns dict with chapter, category, subcategory, detail, and formatted code.
    """
    code = str(code).strip().upper().replace('.', '')
    if not re.match(r'^[A-Z][0-9]{2}', code):
        return {'valid': False, 'raw': code}

    chapter     = icd10_chapters.get(code[0], 'Unknown')
    category    = code[:3]                          # e.g. E11
    subcategory = code[3] if len(code) > 3 else ''  # e.g. 9
    detail      = code[4:] if len(code) > 4 else '' # e.g. 11 (laterality)

    # Formatted with decimal
    formatted = category + ('.' + code[3:] if len(code) > 3 else '')

    return {
        'valid'      : True,
        'formatted'  : formatted,
        'chapter'    : chapter,
        'category'   : category,
        'subcategory': subcategory,
        'detail'     : detail,
        'specificity': ['3-char','4-char','5-char','6-char','7-char'][min(len(code)-3, 4)]
    }

# Test with a range of codes
test_codes = ['E11.9','I10','N18.3','J18.9','A41.9','M16.11','Z87.891','BADCODE']
for c in test_codes:
    parsed = parse_icd10cm(c)
    if parsed.get('valid'):
        print(f"  {c:10s} → formatted: {parsed['formatted']:10s} | chapter: {parsed['chapter'][:40]}")
    else:
        print(f"  {c:10s} → INVALID CODE")


In [ ]:
# --- 3.3  Batch-parse ICD codes in a DataFrame ---
np.random.seed(42)
N = 300
icd_pool = ['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11',
            'N39.0','Z87.891','R06.00','K21.0','F32.9','I25.10']

df_dx = pd.DataFrame({
    'encounter_id': range(1, N+1),
    'primary_dx'  : np.random.choice(icd_pool, N),
    'secondary_dx': np.random.choice(icd_pool + [''], N),
})

# Parse primary diagnosis codes
parsed_cols = df_dx['primary_dx'].apply(parse_icd10cm).apply(pd.Series)
df_dx = pd.concat([df_dx, parsed_cols.add_prefix('dx_')], axis=1)

print("Diagnosis code frequency by ICD-10 chapter:")
display(
    df_dx.groupby('dx_chapter')
         .agg(n_encounters=('encounter_id','count'))
         .sort_values('n_encounters', ascending=False)
         .head(8)
)


## 4. CPT-4 Code Structure (Procedure Codes)

CPT (Current Procedural Terminology) codes are 5-digit numeric codes used to bill for medical procedures and services.

| Range | Section |
|-------|---------|
| 00100–01999 | Anaesthesia |
| 10004–69990 | Surgery (by body system) |
| 70010–79999 | Radiology |
| 80047–89398 | Pathology & Laboratory |
| 90281–99607 | Medicine & Evaluation/Management |
| 99202–99499 | Office/Outpatient E&M visits |


In [ ]:
# --- 4.1  CPT reference table ---
cpt_reference = pd.DataFrame([
    {'cpt':'99213', 'description':'Office visit, established patient, low complexity', 'category':'E&M','rvu':1.30},
    {'cpt':'99214', 'description':'Office visit, established patient, moderate complexity','category':'E&M','rvu':1.92},
    {'cpt':'99232', 'description':'Subsequent hospital care, moderate complexity',       'category':'E&M','rvu':1.39},
    {'cpt':'36415', 'description':'Routine venipuncture',                                'category':'Lab','rvu':0.17},
    {'cpt':'85025', 'description':'Complete CBC with differential',                      'category':'Lab','rvu':0.00},
    {'cpt':'93000', 'description':'Electrocardiogram, routine, with interpretation',    'category':'Medicine','rvu':0.17},
    {'cpt':'71046', 'description':'Chest X-ray, 2 views',                               'category':'Radiology','rvu':0.22},
    {'cpt':'27447', 'description':'Total knee arthroplasty',                             'category':'Surgery','rvu':20.26},
    {'cpt':'43239', 'description':'EGD with biopsy',                                    'category':'Surgery','rvu':4.32},
    {'cpt':'99285', 'description':'ED visit, high complexity',                           'category':'E&M','rvu':4.00},
])

print("CPT code reference:")
display(cpt_reference)


In [ ]:
# --- 4.2  CPT validator ---
def validate_cpt(code: str) -> dict:
    """Check if a CPT code is a valid 5-digit numeric string."""
    code = str(code).strip()
    is_numeric_5 = bool(re.match(r'^\d{5}$', code))
    code_int = int(code) if is_numeric_5 else None

    if not is_numeric_5:
        return {'valid': False, 'code': code, 'section': 'Unknown', 'notes': 'Not 5 digits'}

    # Assign section
    section = 'Unknown'
    if   100   <= code_int <= 1999  : section = 'Anaesthesia'
    elif 10004 <= code_int <= 69990 : section = 'Surgery'
    elif 70010 <= code_int <= 79999 : section = 'Radiology'
    elif 80047 <= code_int <= 89398 : section = 'Pathology/Lab'
    elif 90281 <= code_int <= 99607 : section = 'Medicine/E&M'

    return {'valid': True, 'code': code, 'section': section, 'notes': ''}

for cpt in ['99213','71046','27447','00100','ABCDE','9921']:
    r = validate_cpt(cpt)
    status = 'VALID' if r['valid'] else 'INVALID'
    print(f"  {cpt:8s} → {status:8s} | section: {r['section']}")


## 5. LOINC Code Structure (Lab & Clinical Observations)

LOINC (Logical Observation Identifiers Names and Codes) standardises laboratory and clinical observations. Each code identifies a specific measurement.

Structure: `NNNNN-C`  (5–6 digit numeric) + (1 digit check digit)

LOINC names follow a 6-part axis:
1. **Component** — what was measured (e.g. Glucose)
2. **Property** — type of quantity (e.g. MCnc = mass concentration)
3. **Time aspect** — Pt (point in time) vs 24H (24-hour)
4. **System** — specimen (e.g. Ser/Plas = serum/plasma)
5. **Scale** — Qn (quantitative), Ord (ordinal), Nom (nominal)
6. **Method** — analytical method (if relevant)


In [ ]:
# --- 5.1  LOINC reference for common lab panels ---
loinc_ref = pd.DataFrame([
    {'loinc':'2345-7',  'component':'Glucose',              'property':'MCnc', 'system':'Ser/Plas', 'unit':'mg/dL',  'normal_low':70,  'normal_high':100},
    {'loinc':'2160-0',  'component':'Creatinine',           'property':'MCnc', 'system':'Ser/Plas', 'unit':'mg/dL',  'normal_low':0.6, 'normal_high':1.2},
    {'loinc':'2823-3',  'component':'Potassium',            'property':'SCnc', 'system':'Ser/Plas', 'unit':'mEq/L',  'normal_low':3.5, 'normal_high':5.0},
    {'loinc':'2951-2',  'component':'Sodium',               'property':'SCnc', 'system':'Ser/Plas', 'unit':'mEq/L',  'normal_low':136, 'normal_high':145},
    {'loinc':'718-7',   'component':'Hemoglobin',           'property':'MCnc', 'system':'Bld',      'unit':'g/dL',   'normal_low':12.0,'normal_high':17.5},
    {'loinc':'6690-2',  'component':'WBC',                  'property':'NCnc', 'system':'Bld',      'unit':'10^3/uL','normal_low':4.5, 'normal_high':11.0},
    {'loinc':'777-3',   'component':'Platelets',            'property':'NCnc', 'system':'Bld',      'unit':'10^3/uL','normal_low':150, 'normal_high':400},
    {'loinc':'1742-6',  'component':'ALT',                  'property':'CCnc', 'system':'Ser/Plas', 'unit':'U/L',    'normal_low':7,   'normal_high':56},
    {'loinc':'10839-9', 'component':'Troponin I',           'property':'MCnc', 'system':'Ser/Plas', 'unit':'ng/mL',  'normal_low':0,   'normal_high':0.04},
    {'loinc':'4548-4',  'component':'HbA1c',                'property':'MFr',  'system':'Bld',      'unit':'%',      'normal_low':4.0, 'normal_high':5.6},
    {'loinc':'2093-3',  'component':'Total Cholesterol',    'property':'MCnc', 'system':'Ser/Plas', 'unit':'mg/dL',  'normal_low':0,   'normal_high':200},
    {'loinc':'5902-2',  'component':'Prothrombin time (PT)','property':'Time', 'system':'PPP',      'unit':'s',      'normal_low':11,  'normal_high':13.5},
])

print(f"LOINC reference — {len(loinc_ref)} common labs:")
display(loinc_ref[['loinc','component','unit','normal_low','normal_high']])


In [ ]:
# --- 5.2  Simulate lab results and flag abnormals ---
np.random.seed(7)
n_patients = 200

# For each patient, generate realistic lab values
def sim_lab(loinc_row, n):
    lo, hi = loinc_row['normal_low'], loinc_row['normal_high']
    mid  = (lo + hi) / 2
    span = (hi - lo) / 2
    vals = np.random.normal(mid, span * 0.6, n)
    return vals.clip(lo * 0.4, hi * 2.5).round(2)

lab_records = []
for _, row in loinc_ref.iterrows():
    vals = sim_lab(row, n_patients)
    for i, v in enumerate(vals):
        flag = 'H' if v > row['normal_high'] else ('L' if v < row['normal_low'] else 'N')
        lab_records.append({
            'patient_id' : f'PT-{str(i+1).zfill(5)}',
            'loinc'      : row['loinc'],
            'component'  : row['component'],
            'value'      : v,
            'unit'       : row['unit'],
            'flag'        : flag,
            'normal_low' : row['normal_low'],
            'normal_high': row['normal_high'],
        })

df_labs = pd.DataFrame(lab_records)
print(f"Lab dataset: {len(df_labs):,} results ({n_patients} patients × {len(loinc_ref)} labs)")
print()
# Abnormal rate by test
ab_rate = (df_labs.groupby('component')['flag']
              .apply(lambda x: (x != 'N').mean() * 100)
              .round(1)
              .sort_values(ascending=False)
              .rename('abnormal_pct'))
print("Abnormal rate by lab:")
display(ab_rate.to_frame())


In [ ]:
# --- 5.3  Wide-format lab panel per patient (pivot) ---
lab_wide = df_labs.pivot_table(
    index='patient_id',
    columns='component',
    values='value',
    aggfunc='first'
).reset_index()

print(f"Wide lab panel shape: {lab_wide.shape}")
display(lab_wide.head(5))


## 6. FHIR-Like JSON Parsing

In [ ]:
# FHIR (Fast Healthcare Interoperability Resources) is the modern HL7 standard
# for exchanging health data. Resources are JSON objects. Here we simulate
# a small FHIR Bundle containing Patient + Observation resources.

def make_fhir_bundle(n=5):
    """Generate a minimal FHIR-like bundle for n patients."""
    entries = []
    loinc_sample = loinc_ref[['loinc','component','unit']].head(3)
    
    for i in range(n):
        pt_id = f'patient-{i+1}'
        # Patient resource
        entries.append({
            'resourceType': 'Patient',
            'id': pt_id,
            'gender': np.random.choice(['male','female']),
            'birthDate': f'{np.random.randint(1940,1985)}-{np.random.randint(1,12):02d}-15',
            'address': [{'postalCode': f'{np.random.randint(10000,99999)}'}]
        })
        # Observation resources (labs)
        for _, lab in loinc_sample.iterrows():
            val = round(float(np.random.uniform(
                loinc_ref.loc[loinc_ref['loinc']==lab['loinc'],'normal_low'].values[0],
                loinc_ref.loc[loinc_ref['loinc']==lab['loinc'],'normal_high'].values[0]
            )), 2)
            entries.append({
                'resourceType': 'Observation',
                'id': f'obs-{pt_id}-{lab["loinc"]}',
                'subject': {'reference': f'Patient/{pt_id}'},
                'code': {'coding': [{'system': 'http://loinc.org', 'code': lab['loinc'], 'display': lab['component']}]},
                'valueQuantity': {'value': val, 'unit': lab['unit']},
                'status': 'final'
            })
    return {'resourceType': 'Bundle', 'type': 'collection', 'entry': entries}

bundle = make_fhir_bundle(n=5)

# --- Parse the bundle into tidy DataFrames ---
patients = [e for e in bundle['entry'] if e['resourceType']=='Patient']
observations = [e for e in bundle['entry'] if e['resourceType']=='Observation']

df_pts = pd.DataFrame([{
    'patient_id': p['id'],
    'gender'    : p['gender'],
    'birthDate' : p['birthDate'],
    'zip'       : p['address'][0]['postalCode']
} for p in patients])

df_obs = pd.DataFrame([{
    'patient_id': o['subject']['reference'].split('/')[1],
    'loinc'     : o['code']['coding'][0]['code'],
    'test_name' : o['code']['coding'][0]['display'],
    'value'     : o['valueQuantity']['value'],
    'unit'      : o['valueQuantity']['unit'],
    'status'    : o['status'],
} for o in observations])

print("FHIR Patients:")
display(df_pts)
print("\nFHIR Observations (first 6):")
display(df_obs.head(6))


## 7. HIPAA Safe-Harbour De-identification

In [ ]:
# HIPAA Safe Harbour requires removal of 18 identifiers before data sharing.
# This function demonstrates the safe-harbour checks on a DataFrame.

HIPAA_IDENTIFIERS = [
    'name', 'geographic_data_smaller_than_state', 'dates_except_year',
    'phone', 'fax', 'email', 'ssn', 'mrn', 'health_plan_id',
    'account_number', 'certificate_license', 'vehicle_id', 'device_id',
    'url', 'ip_address', 'biometric', 'full_face_photo', 'any_unique_id'
]

# Simulate a raw dataset with some PHI columns
df_raw = df_pts.copy()
df_raw['full_name']    = [f'Patient {i}' for i in range(len(df_raw))]
df_raw['phone']        = ['555-' + str(np.random.randint(1000000,9999999)) for _ in range(len(df_raw))]
df_raw['email']        = [f'pt{i}@example.com' for i in range(len(df_raw))]
df_raw['ssn_last4']    = [str(np.random.randint(1000,9999)) for _ in range(len(df_raw))]

def safe_harbour_check(df):
    """Flag columns that may contain PHI based on name heuristics."""
    phi_keywords = ['name','phone','fax','email','ssn','mrn','dob','birth',
                    'address','zip','ip','device','url','id_','npi']
    flagged = []
    for col in df.columns:
        if any(kw in col.lower() for kw in phi_keywords):
            flagged.append(col)
    return flagged

flagged_cols = safe_harbour_check(df_raw)
print("Columns flagged as potential PHI:")
for col in flagged_cols:
    print(f"  ⚠  {col}")

# De-identify: drop PHI, generalise ZIP to 3-digit
def deidentify(df, phi_cols, zip_col=None):
    df_clean = df.drop(columns=[c for c in phi_cols if c in df.columns], errors='ignore')
    if zip_col and zip_col in df_clean.columns:
        # HIPAA: 3-digit ZIP if population > 20,000; else replace with 000
        df_clean[zip_col] = df_clean[zip_col].str[:3] + 'XX'
    return df_clean

df_deid = deidentify(df_raw, flagged_cols, zip_col='zip')
print("\nDe-identified dataset:")
display(df_deid)


## 8. Code Quality Validation Pipeline

In [ ]:
# Build a reusable validation function that checks ICD, CPT, and LOINC
# codes in a DataFrame and produces a quality report.

def validate_code_column(series: pd.Series, code_type: str) -> pd.DataFrame:
    """
    Validate a column of clinical codes.
    code_type: 'icd10', 'cpt', or 'loinc'
    """
    results = []
    for code in series:
        if pd.isnull(code) or code == '':
            results.append({'code': code, 'valid': False, 'reason': 'Missing/empty'})
            continue
        code = str(code).strip()
        if code_type == 'icd10':
            r = parse_icd10cm(code)
            results.append({'code': code, 'valid': r['valid'],
                           'reason': '' if r['valid'] else 'Invalid ICD-10 format'})
        elif code_type == 'cpt':
            r = validate_cpt(code)
            results.append({'code': code, 'valid': r['valid'],
                           'reason': r.get('notes','')})
        elif code_type == 'loinc':
            valid = bool(re.match(r'^\d{4,5}-\d$', str(code)))
            results.append({'code': code, 'valid': valid,
                           'reason': '' if valid else 'Invalid LOINC format (expected NNNNN-C)'})
    return pd.DataFrame(results)

# --- Test with mixed valid/invalid codes ---
test_icd = pd.Series(['E11.9','I10','BADCODE','','Z87.891','XYZ'])
test_cpt = pd.Series(['99213','27447','1234','ABCDE','71046'])
test_loinc = pd.Series(['2345-7','718-7','99999','loinc_bad','4548-4'])

for name, series, ctype in [
    ('ICD-10', test_icd, 'icd10'),
    ('CPT', test_cpt, 'cpt'),
    ('LOINC', test_loinc, 'loinc'),
]:
    result = validate_code_column(series, ctype)
    valid_pct = result['valid'].mean() * 100
    print(f"{name} — valid: {valid_pct:.0f}%")
    display(result)
    print()


## 9. Knowledge Check

1. What is the difference between ICD-10-CM and ICD-10-PCS?
2. Why does LOINC exist separately from ICD-10? What does it code for?
3. Which public dataset would you use to study national trends in sepsis hospitalisations — and why?
4. Under HIPAA Safe Harbour, which fields must be removed from a dataset before sharing?
5. Write code to count how many lab results in `df_labs` have a 'H' (high) flag for Glucose and Creatinine.


In [ ]:
# --- Exercise 5 solution ---
high_flags = df_labs[
    df_labs['component'].isin(['Glucose','Creatinine']) & (df_labs['flag'] == 'H')
]
print("High-flagged results by component:")
display(high_flags.groupby('component').size().rename('n_high'))


---
## Summary

In this notebook you:
- Mapped the major US health data ecosystems by access level and key variables
- Built an ICD-10-CM parser that decodes chapter, category, subcategory, and clinical detail
- Validated CPT-4 codes and mapped them to their billing sections
- Constructed a LOINC reference panel and simulated abnormal lab result flagging
- Parsed a FHIR-like JSON bundle into tidy `Patient` and `Observation` DataFrames
- Applied HIPAA Safe-Harbour de-identification heuristics to a raw dataset
- Built a reusable code-quality validation pipeline for ICD, CPT, and LOINC

**You have now completed Module 01.** Proceed to **Module 02 — EDA in Healthcare** for visualisation, time-series analysis, and missingness handling.
